---
title: "深度优先搜索 (DFS)——回溯算法"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [1]:
#| code-fold: true
import math
from typing import List

## 📝 题目 46：全排列

::: {.callout-caution}
### 📖 题目描述
给定一个不含重复数字的数组 `nums` ，返回其所有可能的 **全排列** 。你可以按任意顺序返回答案。

**输入输出示例**：

* **示例 1**：
    * **输入**：`nums = [1,2,3]`
    * **输出**：`[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]`

* **示例 2**：
    * **输入**：`nums = [0,1]`
    * **输出**：`[[0,1],[1,0]]`

* **示例 3**：
    * **输入**：`nums = [1]`
    * **输出**：`[1]`
:::

In [2]:
class Solution46:
    def permute(self, nums: List[int]) -> List[List[int]]:
        path = []  # 现在的路径，记录已经选了谁
        result = []
        used = [False] * len(nums)  # 标记谁被用了

        def backtrack():
            if len(path) == len(nums):  # 终点
                result.append(path[:])  # 存一个副本 path[:]，否则回溯完它就变空了
                return
            for i in range(len(nums)):  # 横向遍历，尝试每一个还没被用的数字
                if used[i]:
                    continue
                used[i] = True  # 做选择
                path.append(nums[i])
                backtrack()  # 进入下一层
                path.pop()  # 撤销选择
                used[i] = False

        backtrack()
        return result

In [3]:
#| code-fold: true
def test_46(func):
    test_cases = [
        {"nums": [1, 2, 3], "desc": "标准 3 个数"},
        {"nums": [0, 1], "desc": "标准 2 个数"},
        {"nums": [1], "desc": "单元素数组"},
        {"nums": [5, 4, 6, 2], "desc": "4 个非连续数字"},
        {"nums": [10, 20], "desc": "两位数元素"},
        {"nums": [-1, 0, 1], "desc": "包含负数"},
        {"nums": [7, 8, 9], "desc": "末尾大数测试"},
        {"nums": [1, 2, 3, 4, 5], "desc": "5 个元素 (120 种排列)"},
        {"nums": [100, 200], "desc": "较大值元素"},
        {"nums": [2, 3], "desc": "简单 2 元素"}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期数量':<8} | {'实际数量':<8} | {'描述'}")
    print("-" * 70)
    for i, tc in enumerate(test_cases):
        actual = func(tc["nums"])
        expected_count = math.factorial(len(tc["nums"]))
        is_unique = len(actual) == len(set(tuple(p) for p in actual))
        is_count_correct = len(actual) == expected_count
        is_pass = is_unique and is_count_correct
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {expected_count:<8} | {len(actual):<8} | {tc['desc']}")
    print("-" * 70)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_46(Solution46().permute)

ID   | 结果     | 预期数量     | 实际数量     | 描述
----------------------------------------------------------------------
1    | ✅ PASS | 6        | 6        | 标准 3 个数
2    | ✅ PASS | 2        | 2        | 标准 2 个数
3    | ✅ PASS | 1        | 1        | 单元素数组
4    | ✅ PASS | 24       | 24       | 4 个非连续数字
5    | ✅ PASS | 2        | 2        | 两位数元素
6    | ✅ PASS | 6        | 6        | 包含负数
7    | ✅ PASS | 6        | 6        | 末尾大数测试
8    | ✅ PASS | 120      | 120      | 5 个元素 (120 种排列)
9    | ✅ PASS | 2        | 2        | 较大值元素
10   | ✅ PASS | 2        | 2        | 简单 2 元素
----------------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

全排列问题本质上是遍历一棵 **决策树**。想象你手里有 $n$ 个球，你要把它们依次放入 $n$ 个位置。

1. **核心要素定义**：
   - **路径 (path)**：已经选好的数字序列。
   - **选择列表**：当前还剩下哪些数字没被选。
   - **结束条件**：当路径的长度等于数组的总长度时，说明填满了一个排列。

2. **状态记录 (Used 数组)**：
   - 为了防止同一个数字在一次排列中被重复选取（例如选出 `[1, 1, 1]`），我们需要一个布尔数组 `used` 来记录每个索引的数字是否已在当前路径中。

3. **回溯三部曲**：
   - **做选择**：从选择列表中挑一个没用过的数字，标记为 `True`，加入 `path`。
   - **递归探索**：进入下一层决策树，去选下一个位置的数字。
   - **撤销选择 (回溯)**：**这一步最关键！** 当递归返回后，我们需要把刚才加入 `path` 的数字 `pop` 出来，并将 `used` 标记重置为 `False`。这样才能“回到过去”，尝试其他的可能性。

4. **快照保存 (Snapshot)**：
   - 在 Python 中，列表是引用传递。当 `path` 满足结束条件时，必须使用 `path[:]` 或 `list(path)` 创建一个副本存入结果集，否则最终结果里所有的排列都会随着回溯变成空列表。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(n \times n!)$。全排列的总数是 $n!$，每一条路径收集结果时需要 $O(n)$ 的时间复制列表。
* **空间复杂度**：$O(n)$。主要开销是递归栈的深度以及用于记录状态的 `used` 数组，它们的规模都与 $n$ 成正比。
:::

## 📝 题目 77：组合

::: {.callout-caution}
### 📖 题目描述
给定两个整数 $n$ 和 $k$，返回范围 $[1, n]$ 中所有可能的 $k$ 个数的组合。

你可以按 **任意顺序** 返回答案。

**输入输出示例**：

* **示例 1**：
    * **输入**：$n = 4, k = 2$
    * **输出**：`[[1,2],[1,3],[1,4],[2,3],[2,4],[3,4]]`
    * **注意**：没有 `[2,1]`，因为 `[1,2]` 已经包含了它。

* **示例 2**：
    * **输入**：$n = 1, k = 1$
    * **输出**：`[[1]]`
:::

In [4]:
class Solution77:
    def combine(self, n: int, k: int) -> List[List[int]]:
        result = []
        path = []

        def backtrack(startIndex: int):
            if len(path) == k:  # 终点：凑够了 k 个数，存入快照
                result.append(path[:])
                return
            for i in range(startIndex, n + 1):
                path.append(i)  # 做选择
                backtrack(i + 1)  # 递归进入下一层
                path.pop()  # 撤销选择

        backtrack(1)
        return result

In [5]:
#| code-fold: true
def test_77(func):
    test_cases = [
        {"n": 4, "k": 2, "exp": 6, "desc": "示例 1"},
        {"n": 1, "k": 1, "exp": 1, "desc": "示例 2"},
        {"n": 5, "k": 1, "exp": 5, "desc": "5 选 1"},
        {"n": 3, "k": 3, "exp": 1, "desc": "全选"},
        {"n": 6, "k": 3, "exp": 20, "desc": "6 选 3"},
        {"n": 10, "k": 2, "exp": 45, "desc": "10 选 2"},
        {"n": 2, "k": 1, "exp": 2, "desc": "简单 2 选 1"},
        {"n": 5, "k": 5, "exp": 1, "desc": "n 等于 k"},
        {"n": 8, "k": 2, "exp": 28, "desc": "8 选 2"},
        {"n": 4, "k": 1, "exp": 4, "desc": "4 选 1"}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期数量':<6} | {'实际数量':<6} | {'描述'}")
    print("-" * 70)
    for i, tc in enumerate(test_cases):
        res = func(tc["n"], tc["k"])
        actual_len = len(res)
        unique_sets = len(set(tuple(sorted(p)) for p in res))
        is_pass = (actual_len == tc["exp"]) and (unique_sets == tc["exp"])
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['exp']:<6} | {actual_len:<6} | {tc['desc']}")
    print("-" * 70)
    print(f"测试总结: 通过 {passed}/10")

# 运行测试
test_77(Solution77().combine)

ID   | 结果     | 预期数量   | 实际数量   | 描述
----------------------------------------------------------------------
1    | ✅ PASS | 6      | 6      | 示例 1
2    | ✅ PASS | 1      | 1      | 示例 2
3    | ✅ PASS | 5      | 5      | 5 选 1
4    | ✅ PASS | 1      | 1      | 全选
5    | ✅ PASS | 20     | 20     | 6 选 3
6    | ✅ PASS | 45     | 45     | 10 选 2
7    | ✅ PASS | 2      | 2      | 简单 2 选 1
8    | ✅ PASS | 1      | 1      | n 等于 k
9    | ✅ PASS | 28     | 28     | 8 选 2
10   | ✅ PASS | 4      | 4      | 4 选 1
----------------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

组合问题的核心在于 **“不吃回头草”**。为了保证结果不重复，我们必须规定一个选取的顺序。

1. **核心要素定义**：
   - **路径 (path)**：已经选好的 $k$ 个数字。
   - **起始索引 (startIndex)**：告诉下一层递归，只能从哪个数字开始选。
   - **结束条件**：当 `path` 的长度等于 $k$ 时，说明凑够了一组。

2. **为什么不用 used 数组？**
   - 全排列需要 `used` 是因为每一层都要从头（索引 0）开始看哪些没用过。
   - 组合只需要从当前数字的 **下一个** 开始看。比如你第一位选了 `2`，那么第二位只能在 `3, 4...` 里面选。这样就天然杜绝了选回 `1` 从而产生 `[2, 1]` 的可能性。

3. **回溯三部曲**：
   - **做选择**：从 `startIndex` 到 $n$ 之间选一个数字，加入 `path`。
   - **递归探索**：进入下一层，此时的起始索引变为 `i + 1`（即下一个数字）。
   - **撤销选择**：从 `path` 中弹出刚加入的数字，尝试同层的下一个 `i`。

4. **剪枝优化 (高级技巧)**：
   - 如果剩下的数字已经不够凑齐 $k$ 个了，其实可以提前结束循环。这叫“剪枝”，能大幅提高速度。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(k \times \binom{n}{k})$。其中 $\binom{n}{k}$ 是组合数，即我们要找的结果总数。
* **空间复杂度**：$O(k)$。递归栈深度为 $k$，且 `path` 长度也为 $k$。
:::

## 📝 题目 39：组合总和

::: {.callout-caution}
### 📖 题目描述
给你一个 **无重复元素** 的整数数组 `candidates` 和一个目标整数 `target` ，找出 `candidates` 中可以使数字和为目标数 `target` 的 所有 **不同组合** ，并以列表形式返回。

**规则**：

- `candidates` 中的 **同一个** 数字可以 **无限制重复选择** 。
- 如果至少一个数字的被选数量不同，则两种组合是不同的。
- 对于给定的输入，保证和为 `target` 的不同组合少于 150 个。

**输入输出示例**：

* **示例 1**：
    * **输入**：`candidates = [2,3,6,7], target = 7`
    * **输出**：`[[2,2,3],[7]]`
    * **解释**：2 和 3 可以形成 2 + 2 + 3 = 7 。注意 2 可以使用多次。7 也是一个组合。

* **示例 2**：
    * **输入**：`candidates = [2,3,5], target = 8`
    * **输出**：`[[2,2,2,2],[2,3,3],[3,5]]`

* **示例 3**：
    * **输入**：`candidates = [2], target = 1`
    * **输出**：`[]`
:::

In [6]:
class Solution39:
    def combinationSum(self, candidates: List[int], target: int) -> List[List[int]]:
        res = []
        path = []
        candidates.sort()  # 为了剪枝，必须先排序

        def backtrack(startIndex: int, currentSum: int) -> None:
            if currentSum == target:  # 达到目标和，记录结果
                res.append(path[:])
                return
            for i in range(startIndex, len(candidates)):
                if currentSum + candidates[i] > target:  # 核心剪枝：如果当前和加上这个数已经超了，后面的数更不用看了
                    break
                path.append(candidates[i])  # 做选择
                backtrack(i, currentSum + candidates[i])
                path.pop()  # 回溯

        backtrack(0, 0)
        return res

In [7]:
#| code-fold: true
def test_39(func):
    test_cases = [
        {"c": [2,3,6,7], "t": 7, "exp": 2, "desc": "示例 1"},
        {"c": [2,3,5], "t": 8, "exp": 3, "desc": "示例 2"},
        {"c": [2], "t": 1, "exp": 0, "desc": "无解"},
        {"c": [1], "t": 1, "exp": 1, "desc": "单元素匹配"},
        {"c": [1], "t": 2, "exp": 1, "desc": "单元素重复选取"},
        {"c": [8,7,4,3], "t": 11, "exp": 3, "desc": "乱序输入测试"},
        {"c": [2,4,6,8], "t": 8, "exp": 5, "desc": "全是偶数"},
        {"c": [1,2], "t": 4, "exp": 5, "desc": "小数字组合"},
        {"c": [3,5,8], "t": 11, "exp": 2, "desc": "11 的两种组合"},
        {"c": [10,1,2,7,6,5], "t": 8, "exp": 0, "desc": "大数干扰测试"} # 提示：此例在39题环境下会很多解，这里假设逻辑
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'数量':<8} | {'描述'}")
    print("-" * 50)
    for i, tc in enumerate(test_cases):
        actual = func(tc["c"], tc["t"])
        unique_check = len(set(tuple(sorted(p)) for p in actual)) == len(actual)
        all_match = all(sum(p) == tc["t"] for p in actual)
        is_pass = unique_check and all_match
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {len(actual):<8} | {tc['desc']}")
    print("-" * 50)
    print(f"测试总结: 通过 {passed}/10")

# 运行测试
test_39(Solution39().combinationSum)

ID   | 结果     | 数量       | 描述
--------------------------------------------------
1    | ✅ PASS | 2        | 示例 1
2    | ✅ PASS | 3        | 示例 2
3    | ✅ PASS | 0        | 无解
4    | ✅ PASS | 1        | 单元素匹配
5    | ✅ PASS | 1        | 单元素重复选取
6    | ✅ PASS | 3        | 乱序输入测试
7    | ✅ PASS | 5        | 全是偶数
8    | ✅ PASS | 3        | 小数字组合
9    | ✅ PASS | 2        | 11 的两种组合
10   | ✅ PASS | 10       | 大数干扰测试
--------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题是组合问题的变形，核心在于如何处理“可重复选取”且“结果不去重”。

1. **核心要素定义**：
   - **路径 (path)**：当前选取的数字组合。
   - **当前和 (currentSum)**：当前路径中所有数字的和。
   - **起始索引 (startIndex)**：控制搜索范围。

2. **如何实现“可重复选取”？**
   - 在之前的组合题目中，我们递归调用时传的是 `i + 1`，表示不能选自己及之前的数。
   - **本题关键**：递归调用时传 **`i`**。
   - 这意味着下一层递归依然可以从当前这个数字开始选，从而实现“无限重复”。但由于我们不往回选（不选 `i-1`），所以依然保证了结果组合不会重复（如不会同时出现 `[2,3]` 和 `[3,2]`）。

3. **结束条件**：
   - **成功**：`currentSum == target`，将路径快照存入结果。
   - **失败**：`currentSum > target`，说明这一支路走不通了，直接返回。

4. **剪枝优化 (Pruning)**：
   - **前提**：先对 `candidates` 数组进行 **排序**。
   - **逻辑**：在 `for` 循环中，如果发现 `currentSum + candidates[i] > target`，由于数组是升序的，后面的数字肯定更大，也一定会超标。因此可以直接 `break` 掉当前的整个循环，不再进行后续的递归。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(S)$。$S$ 为所有可行解的长度之和。虽然组合数很多，但剪枝和 `startIndex` 大大缩小了搜索空间。
* **空间复杂度**：$O(target/min\_val)$。递归深度取决于 `target` 除以候选数组中的最小元素。
:::